In [2]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor

from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler

df = pd.read_csv("../data/subjects.csv")
print(df.shape)

In [3]:
# 피처 선택

# 1단계 : y2(호감 강도)와 상관 0.5 이상 남기기

corr = df.select_dtypes(include='number').corr()
target_corr = corr['y2_attraction_intensity'].drop(['y1_attracted', 'y2_attraction_intensity', 'y3_dated', 'y4_relationship_satisfaction']).abs()

strong = target_corr[target_corr >= 0.5]

print(f'1단계 : {len(target_corr)}개 -> {len(strong)}개 (y2와 관련 있는 것만)')

# 2단계 : 피처끼리 상관 0.8 이상이면 하나 제거로 다중공선성 제거
strong_corr = df[strong.index].corr()
mask = np.triu(np.ones(strong_corr.shape), k=1).astype(bool)
pairs = strong_corr.where(mask).stack()
high_pairs = pairs[pairs >= 0.8]


to_drop = set()

for (a, b), val in high_pairs.items():
    if abs(target_corr[a]) >= abs(target_corr[b]):
        to_drop.add(b)
    else:
        to_drop.add(a)

selected = [c for c in strong.index if c not in to_drop]

print(f'2단계 : {len(strong)}개 => {len(selected)}개 (중복제거)')


In [4]:
# 셀 4: 이상형 제조기 — 모델 학습 (selected 12개 전체 사용)

X = df[selected]
y = df['y2_attraction_intensity']

model = XGBRegressor(n_estimators=100, random_state=42)
model.fit(X,y)

print(f'피처 {len(selected)}개로 학습 완료')
print(f'y2 범위 : {y.min()} ~ {y.max()}')

In [5]:
# 셀 5: 이상형 프로필 역산
# "y2 >= 8이 되려면 피처 조합이 어때야 하나?"

# 1) 가상 인물 10000명 생성 (넓게 탐색)
np.random.seed(42)
n_sim = 10000
means = df[selected].mean()
stds = df[selected].std()

simulated = pd.DataFrame(
    np.random.normal(loc=means, scale=stds, size=(n_sim, len(selected))),
    columns=selected
).clip(1, 10)

pred = model.predict(simulated)

# 2) 이상형(8+) vs 비이상형 비교
simulated['pred_y2'] = pred
ideal = simulated[simulated['pred_y2'] >= 8]
not_ideal = simulated[simulated['pred_y2'] < 8]

print(f'10000명 중 이상형: {len(ideal)}명 ({len(ideal)/100:.1f}%)')
print(f'\n=== 이상형의 평균 피처 프로필 ===')
profile = ideal[selected].mean().sort_values(ascending=False)
print(profile.round(1))

In [7]:
# 셀 6: 이상형 프로필에 가장 가까운 실제 사람 찾기
from scipy.spatial.distance import euclidean

# 이상형 평균 프로필
ideal_profile = ideal[selected].mean()

# 실제 22명과의 거리 계산
distances = df.apply(
    lambda row: euclidean(row[selected], ideal_profile), axis=1
)
df['ideal_distance'] = distances

# 가장 가까운 사람
closest_idx = distances.idxmin()
closest = df.loc[closest_idx]

print(f'=== 이상형에 가장 가까운 사람: {closest["name_or_code"]} ===')
print(f'거리: {distances[closest_idx]:.2f}\n')

# 잔차 계산 — 숫자만 확실히
residual = (ideal_profile - closest[selected].astype(float)).sort_values(ascending=False)
print('=== 잔차 (이상형 대비 부족/초과) ===')
print(residual.round(1))

In [8]:
# 셀 7: 이상형 제조기 시각화

import matplotlib.pyplot as plt
import platform
if platform.system() == 'Darwin':
    plt.rcParams['font.family'] = 'AppleGothic'
elif platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

feat_kr = {
    'E34_push_pull_tension': '밀당 텐션',
    'G45_conversation_fun': '대화 재미',
    'B5_appearance': '외모',
    'B7_voice': '목소리',
    'B9_scent_sensory': '향/감각',
    'D21_humor_match': '유머 코드',
    'G42_complementary_competence': '상호보완',
    'D20_cognitive_sharpness': '지적 예리함',
    'D24_clumsy_but_trying': '서툰 노력',
    'E32_care_impulse': '돌봄 충동',
    'H51_social_barrier': '사회적 장벽',
    'D17_honesty_transparency': '솔직함'
}

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# --- 왼쪽: 이상형 프로필 ---
ax1 = axes[0]
labels = [feat_kr.get(f, f) for f in profile.index]
colors = ['#e74c3c' if v >= 7 else '#3498db' if v >= 5 else '#95a5a6' for v in profile.values]
bars = ax1.barh(labels, profile.values, color=colors)
for bar, val in zip(bars, profile.values):
    ax1.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2,
             f'{val:.1f}', va='center', fontsize=10)
ax1.set_xlim(0, 10)
ax1.set_xlabel('평균 점수')
ax1.set_title('당신의 이상형 프로필', fontsize=13, fontweight='bold')
ax1.invert_yaxis()

# --- 오른쪽: 가장 가까운 사람 vs 이상형 잔차 ---
ax2 = axes[1]
res_labels = [feat_kr.get(f, f) for f in residual.index]
res_colors = ['#2ecc71' if v > 0 else '#e67e22' for v in residual.values]
ax2.barh(res_labels, residual.values, color=res_colors)
for i, (val, label) in enumerate(zip(residual.values, res_labels)):
    offset = 0.15 if val >= 0 else -0.15
    ha = 'left' if val >= 0 else 'right'
    ax2.text(val + offset, i, f'{val:+.1f}', va='center', ha=ha, fontsize=10)
ax2.axvline(0, color='black', linewidth=0.8)
ax2.set_title(f'이상형 대비 잔차: {closest["name_or_code"]}', fontsize=13, fontweight='bold')
ax2.invert_yaxis()

plt.suptitle('이상형 제조기', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('ideal_type_generator.png', dpi=150, bbox_inches='tight')
plt.show()

In [14]:
# 셀 8 (선택): 이상형에 가까운 TOP 3
top3 = distances.nsmallest(3)
for rank, (idx, dist) in enumerate(top3.items(), 1):
    name = df.loc[idx, 'name_or_code']
    y2 = df.loc[idx, 'y2_attraction_intensity']
    print(f'{rank}위: {name} (거리: {dist:.2f}, 실제 y2: {y2})')

In [13]:
# 심의 피처 vs 이상형 프로필 비교
sim_row = df[df['name_or_code'] == 'Subject #02'].iloc[0]

comparison = pd.DataFrame({
    '이상형': ideal_profile,
    'Subject #02': sim_row[selected].astype(float),
    '잔차': ideal_profile - sim_row[selected].astype(float)
}).round(1)

print(f'심의 거리: {distances[df[df["name_or_code"]=="Subject #02"].index[0]]:.2f}')
print(f'심의 실제 y2: {sim_row["y2_attraction_intensity"]}')
print(f'\n{comparison.sort_values("잔차", ascending=False)}')

In [15]:
# 거리 대신 — 예측 y2가 높은 순으로 정렬
df['pred_y2'] = model.predict(df[selected])
print(df[['name_or_code','pred_y2','y2_attraction_intensity','ideal_distance']]
      .sort_values('pred_y2', ascending=False).round(2))

In [16]:
# 셀 8: 연애 자아분석 — 피처 중요도 시각화

from xgboost import XGBRegressor
import matplotlib.pyplot as plt

# 모델에서 importance 추출
importance = pd.DataFrame({
    'feature': selected,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

# 한글 매핑
feat_kr = {
    'E34_push_pull_tension': '밀당 텐션',
    'G45_conversation_fun': '대화 재미',
    'B5_appearance': '외모',
    'B7_voice': '목소리',
    'B9_scent_sensory': '향/감각',
    'D21_humor_match': '유머 코드',
    'G42_complementary_competence': '상호보완',
    'D20_cognitive_sharpness': '지적 예리함',
    'D24_clumsy_but_trying': '서툰 노력',
    'E32_care_impulse': '돌봄 충동',
    'H51_social_barrier': '사회적 장벽',
    'D17_honesty_transparency': '솔직함'
}

importance['label'] = importance['feature'].map(feat_kr)
importance['pct'] = (importance['importance'] / importance['importance'].sum() * 100).round(1)

# 시각화
fig, ax = plt.subplots(figsize=(10, 6))
colors = ['#e74c3c' if i < 3 else '#3498db' if i < 6 else '#bdc3c7' 
          for i in range(len(importance))]
bars = ax.barh(importance['label'], importance['pct'], color=colors)

for bar, pct in zip(bars, importance['pct']):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
            f'{pct}%', va='center', fontsize=11, fontweight='bold')

ax.set_xlabel('끌림 기여도 (%)')
ax.set_title('당신의 끌림을 결정하는 것들', fontsize=14, fontweight='bold')
ax.invert_yaxis()

plt.tight_layout()
plt.savefig('self_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

In [17]:
# 셀 8: 연애 자아분석 — 인포그래픽 스타일

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# importance 추출
importance = pd.DataFrame({
    'feature': selected,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

feat_kr = {
    'E34_push_pull_tension': '밀당 텐션',
    'G45_conversation_fun': '대화 재미',
    'B5_appearance': '외모',
    'B7_voice': '목소리',
    'B9_scent_sensory': '향/감각',
    'D21_humor_match': '유머 코드',
    'G42_complementary_competence': '상호보완',
    'D20_cognitive_sharpness': '지적 예리함',
    'D24_clumsy_but_trying': '서툰 노력',
    'E32_care_impulse': '돌봄 충동',
    'H51_social_barrier': '사회적 장벽',
    'D17_honesty_transparency': '솔직함'
}

feat_icon = {
    'E34_push_pull_tension': '🔥',
    'G45_conversation_fun': '💬',
    'B5_appearance': '👀',
    'B7_voice': '🎵',
    'B9_scent_sensory': '🌸',
    'D21_humor_match': '😂',
    'G42_complementary_competence': '🤝',
    'D20_cognitive_sharpness': '🧠',
    'D24_clumsy_but_trying': '🥺',
    'E32_care_impulse': '💗',
    'H51_social_barrier': '🚧',
    'D17_honesty_transparency': '🪞'
}

importance['label'] = importance['feature'].map(feat_kr)
importance['icon'] = importance['feature'].map(feat_icon)
importance['pct'] = (importance['importance'] / importance['importance'].sum() * 100).round(1)

# 시각화
fig, ax = plt.subplots(figsize=(12, 8))
fig.patch.set_facecolor('#1a1a2e')
ax.set_facecolor('#1a1a2e')

n = len(importance)
y_pos = np.arange(n)

# 그라데이션 색상
colors = []
for i in range(n):
    if i < 3:
        colors.append(f'#{hex(255 - i*30)[2:]}6B6B')
    elif i < 7:
        colors.append(f'#6B{hex(160 - (i-3)*20)[2:]}CC')
    else:
        colors.append('#4a4a6a')

# 배경 바 (100% 기준)
ax.barh(y_pos, [importance['pct'].max() * 1.3] * n, 
        color='#16213e', height=0.6, zorder=1)

# 실제 바
bars = ax.barh(y_pos, importance['pct'].values, 
               color=colors, height=0.6, zorder=2,
               edgecolor='none')

# 라벨 + 아이콘 + 수치
for i, (_, row) in enumerate(importance.iterrows()):
    # 아이콘 + 한글 이름 (왼쪽)
    ax.text(-1.5, i, f'{row["icon"]}  {row["label"]}', 
            va='center', ha='right', fontsize=13, 
            color='white', fontweight='bold')
    
    # 퍼센트 (바 끝)
    ax.text(row['pct'] + 0.8, i, f'{row["pct"]}%', 
            va='center', ha='left', fontsize=12, 
            color='#e0e0e0', fontweight='bold')
    
    # 순위 뱃지 (상위 3개만)
    if i < 3:
        badge_colors = ['#FFD700', '#C0C0C0', '#CD7F32']
        ax.text(row['pct'] + 4, i, f'#{i+1}', 
                va='center', ha='left', fontsize=10,
                color=badge_colors[i], fontweight='bold',
                bbox=dict(boxstyle='round,pad=0.3', 
                         facecolor='#16213e', edgecolor=badge_colors[i], linewidth=1.5))

# 스타일
ax.set_xlim(-15, importance['pct'].max() * 1.6)
ax.set_yticks([])
ax.set_xticks([])
ax.spines[:].set_visible(False)

# 타이틀
ax.text(-15, -1.2, '🔍 당신의 끌림을 결정하는 것들', 
        fontsize=18, fontweight='bold', color='white')
ax.text(-15, -0.5, 'Feature Importance Analysis — 어떤 특성이 당신의 호감에 가장 큰 영향을 미치는가',
        fontsize=9, color='#888888')

# 하단 요약
top3 = importance.head(3)
summary = ' + '.join([f'{r["icon"]}{r["label"]}' for _, r in top3.iterrows()])
total_pct = top3['pct'].sum()
ax.text(-15, n + 0.3, f'TOP 3 합산: {total_pct:.0f}%  |  {summary}',
        fontsize=11, color='#e0e0e0', style='italic')

plt.tight_layout()
plt.savefig('self_analysis.png', dpi=150, bbox_inches='tight', 
            facecolor='#1a1a2e')
plt.show()